# GymRec — train the exercise classifier and export the best model

Runs Phases 3–4 from `JOURNAL.md` on Kaggle:
**keypoints → dataset → train RF / XGBoost / LSTM / GRU / CNN → pick the best on validation → test → model zip**.

Before running, in the right-hand panel:
* **Add Input**: your video dataset
* **Accelerator**: GPU (T4 or P100)
* **Internet**: On (to clone the repo and download YOLO-pose)

For a full run use **Save Version → Save & Run All**: it runs in the background for up to 12 hours
and keeps `/kaggle/working` as output. At the end, download `gym/gymrec_model_<id>.zip` from the Output tab.

In [ ]:
# 1. Get the code
%cd /kaggle/working
!test -d Gym_Exercise || git clone -q https://github.com/Sarthak-D96/Gym_Exercise.git
%cd /kaggle/working/Gym_Exercise
!git pull -q
!pip install -q ultralytics
import torch; print("GPU:", torch.cuda.is_available())

In [ ]:
# 2. Find your dataset: the folder that contains one sub-folder per exercise
!ls /kaggle/input
!find /kaggle/input -maxdepth 4 -type d | head -60

## 3. Configure

Defaults live in `training/config.py`. Anything set here overrides them for this run.

* **`CLASSES`**: which exercises to train on. Slugs from `config.EXERCISES`, which also lists the
  dataset folder names that map to each slug.
* **`DATA_DIRS` / `TEST_DIRS`**: point them at the folders found above. Leave `TEST_DIRS` empty to split 70/15/15.
* **`MODELS`**: which model types to train. The best one on the validation set is exported.

In [ ]:
import sys; sys.path.insert(0, "training")
import config, run_all

config.CLASSES = ["squat", "push_up", "shoulder_press", "bicep_curl", "bench_press", "pull_up"]

config.DATA_DIRS = ["/kaggle/input/gym-workoutexercises-video/verified_data/verified_data"]   # <- change
config.TEST_DIRS = ["/kaggle/input/gym-workoutexercises-video/test/test"]                     # <- change, or []
config.WORK_DIR = "/kaggle/working/gym"

config.MODELS = ["rf", "xgb", "lstm", "gru", "cnn"]
config.SELECT_BY = "macro_f1"   # chosen on the validation set
# config.FORCE_MODEL = "lstm"   # ship a specific model instead of the best one
# config.GROUP_REGEX = r"^(.*)_\d+$"   # if clips cut from one video share a name prefix

# A dataset folder with a different name? Map it to a class:
# config.EXERCISES["bicep_curl"]["folders"].append("Biceps Curl (dumbbell)")

In [ ]:
# 4. Check which folders feed which class (fix the config above if something is missing)
run_all.preview()

In [ ]:
# 5. Run everything: extract (slow, resumable) -> build -> train -> select & export
run_all.main()
# Re-run only some steps, e.g. after changing MODELS:  run_all.main(["train", "select"])
# After changing CLASSES, extract again (finished videos are skipped):  run_all.main()

In [ ]:
# 6. Results and the model to download
from pathlib import Path
zips = sorted(Path(config.WORK_DIR).glob("gymrec_model_*.zip"))
latest = zips[-1]
print(open(latest.with_suffix("") / "results.md").read())
print("Download from the Output tab:", latest)

## 7. Use it on the website (on the laptop)

```bash
pip install -r requirements.txt            # torch + ultralytics (+ scikit-learn / xgboost if a tree model won)
python manage.py install_model ~/Downloads/gymrec_model_<id>.zip
```

`install_model` unpacks the zip into `ml_models/`, adds any new exercises, activates the model
and its pose weights, and shows only the trained exercises on the website.